In [3]:
push!(LOAD_PATH, "./src")
include("src/CNN.jl")
using Random
using MLDatasets

In [4]:
function onehot(target, classes=0:9)
    num_classes = length(classes)
    encoded = zeros(Float32, num_classes)
    idx = target - first(classes) + 1
    encoded[idx] = 1.0f0
    return encoded
end

function loader(data; batchsize::Int=32)
    x_all = Float32.(reshape(data.features, 28, 28, 1, :))
    targets = data.targets
    
    n_samples = size(x_all, 4)
    indices = randperm(n_samples)
    
    return (
        begin
            batch_idx = indices[i:min(i + batchsize - 1, n_samples)]
            
            x_batch = x_all[:, :, :, batch_idx]
            y_batch = hcat([onehot(targets[idx], 0:9) for idx in batch_idx]...)
            
            (x_batch, y_batch)
        end
        for i in 1:batchsize:n_samples
    )
end

loader (generic function with 1 method)

In [7]:
import .CNN: chain, conv2d, MaxPool, Flatten, dense, relu, Dropout, softmax, GraphNode, CrossEntropy, graph, train_step!, loss_and_accuracy_clean
using MLDatasets
using Random

# Load data
train_mnist = FashionMNIST(split=:train)
test_mnist  = FashionMNIST(split=:test)

train_data = (
    features = train_mnist.features,
    targets  = train_mnist.targets,
)

test_data = (
    features = test_mnist.features,
    targets  = test_mnist.targets,
)

# Define architecture
net = chain((
  conv2d((3, 3), 1 => 6, pad=1, bias=false),
  MaxPool((2, 2)),
  conv2d((3, 3), 6 => 16, pad=1, bias=false),
  MaxPool((2, 2)),
  Flatten(),
  dense(784 => 84, relu),
  Dropout(0.4),
  dense(84 => 10),
  softmax()
))

input_tensor = GraphNode(zeros(Float32, 28, 28, 1))
target_tensor = GraphNode(zeros(Float32, 10))
output_node = net(input_tensor)
loss_node = CrossEntropy()(output_node, target_tensor)
model_graph = graph(loss_node)

settings = (; eta = 0.01f0, epochs = 3, batchsize = 10)
accuracy = zeros(settings.epochs, 2)
train_log = []

# Training loop
for epoch in 1:settings.epochs
    @time for (x_batch, y_batch) in loader(train_data, batchsize=settings.batchsize)
        train_step!(model_graph, input_tensor, target_tensor, x_batch, y_batch, settings.eta)
    end
    
    train_stats = loss_and_accuracy_clean(model_graph, input_tensor, output_node, loss_node, target_tensor, loader(train_data, batchsize=1000))
    test_stats  = loss_and_accuracy_clean(model_graph, input_tensor, output_node, loss_node, target_tensor, loader(test_data, batchsize=1000))
    
    println("[Epoka $epoch] Dokładność (Train): $(train_stats.acc)% | (Test): $(test_stats.acc)%")
    flush(stdout)
    
    push!(train_log, (; epoch, train_stats..., test_stats...))
    accuracy[epoch, 1] = train_stats.acc
    accuracy[epoch, 2] = test_stats.acc
end

  6.503658 seconds (10.74 M allocations: 678.400 MiB, 3.33% gc time)
[Epoka 1] Dokładność (Train): 84.55% | (Test): 83.32%
  6.504906 seconds (10.74 M allocations: 678.400 MiB, 3.33% gc time)
[Epoka 2] Dokładność (Train): 87.64% | (Test): 86.78%
  6.496870 seconds (10.74 M allocations: 678.400 MiB, 3.19% gc time)
[Epoka 3] Dokładność (Train): 88.63% | (Test): 87.6%


# Testing performance

In [ ]:
# import Pkg; Pkg.add("TimerOutputs")
using TimerOutputs

const to = TimerOutput()

────────────────────────────────────────────────────────────────────
                           Time                    Allocations      
                  ───────────────────────   ────────────────────────
Tot / % measured:      139ms /   0.0%           5.52MiB /   0.0%    

Section   ncalls     time    %tot     avg     alloc    %tot      avg
────────────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────────────

In [ ]:
function train_step!(graph, input_node, target_node, x_batch, y_batch, eta)
    @timeit to "1. zerograd" CNN.zerograd!(graph)
    batch_size = size(x_batch, 4)
    
    for i in 1:batch_size
        @timeit to "2. zero_act" CNN.zero_activations_grad!(graph)
        
        @timeit to "3. views" begin
            x_single = @view x_batch[:, :, :, i]
            y_single = @view y_batch[:, i]
        end
        
        @timeit to "4. forward" CNN.forward!(graph, input_node => x_single, target_node => y_single, train_mode=true)
        @timeit to "5. backward" CNN.backward!(graph)
    end
    
    @timeit to "6. optimize" CNN.optimize!(graph, eta / batch_size) 
end

train_step! (generic function with 1 method)

In [ ]:
reset_timer!(to)

x_batch, y_batch = first(loader(train_data, batchsize=10))
train_step!(model_graph, input_tensor, target_tensor, x_batch, y_batch, settings.eta)

show(to)

────────────────────────────────────────────────────────────────────────
                               Time                    Allocations      
                      ───────────────────────   ────────────────────────
  Tot / % measured:        5.95s /  69.6%            686MiB /  48.6%    

Section       ncalls     time    %tot     avg     alloc    %tot      avg
────────────────────────────────────────────────────────────────────────
5. backward       10    1.93s   46.7%   193ms    128MiB   38.2%  12.8MiB
4. forward        10    1.68s   40.7%   168ms    159MiB   47.7%  15.9MiB
6. optimize        1    475ms   11.5%   475ms   44.1MiB   13.2%  44.1MiB
1. zerograd        1   49.0ms    1.2%  49.0ms   3.03MiB    0.9%  3.03MiB
2. zero_act       10   50.3μs    0.0%  5.03μs     0.00B    0.0%    0.00B
3. views          10   1.80μs    0.0%   180ns     0.00B    0.0%    0.00B
────────────────────────────────────────────────────────────────────────

In [8]:
# import Pkg
# Pkg.gc()
# Pkg.add("ProfileSVG")
using Profile
using ProfileSVG

In [ ]:
x_batch, y_batch = first(loader(train_data, batchsize=10))

train_step!(model_graph, input_tensor, target_tensor, x_batch, y_batch, settings.eta)

In [ ]:
Profile.clear()

@profile for i in 1:20
    train_step!(model_graph, input_tensor, target_tensor, x_batch, y_batch, settings.eta)
end

In [ ]:
ProfileSVG.view()
ProfileSVG.save("my_profile.svg")